# Joins

In [3]:
import psycopg2

conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='movies', user='postgres', password='postgres'
)
cursor = conn.cursor()

cursor.execute('DELETE FROM employees')
cursor.execute('DELETE FROM departments')
conn.commit()

for _stmt in '''
    CREATE TABLE IF NOT EXISTS departments (
        id   INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        budget NUMERIC(10,2)
    );
    CREATE TABLE IF NOT EXISTS employees (
        id            INTEGER PRIMARY KEY,
        name          TEXT,
        department_id INTEGER,
        salary        NUMERIC(10,2)
    );
'''.split(';'):
    _s = _stmt.strip()
    if _s and not _s.startswith('--'):
        cursor.execute(_s)
cursor.executemany('INSERT INTO departments VALUES (%s,%s,%s)', [
    (1, 'Engineering', 500000),
    (2, 'Marketing',   200000),
    (3, 'HR',          150000),
    (4, 'Finance',     300000),    # sin empleados asignados
])
cursor.executemany('INSERT INTO employees VALUES (%s,%s,%s,%s)', [
    (1, 'Alice',  1, 75000),
    (2, 'Bob',    2, 55000),
    (3, 'Carol',  1, 82000),
    (4, 'David',  3, 48000),
    (5, 'Eva',    2, 61000),
    (6, 'Frank',  None, 70000),    # sin departamento asignado
])
conn.commit()

# INNER JOIN — solo emp con dept asignado Y dept existente
cursor.execute("""
    SELECT e.name, d.name AS department, e.salary
    FROM employees e
    INNER JOIN departments d ON e.department_id = d.id
    ORDER BY e.name
""")
print("INNER JOIN (solo coincidencias):")
for row in cursor.fetchall():
    print(f"  ${row[0]:<8} | ${row[1]:<12} | $${row[2]:,}")
# Nota: Frank (dept=None) y Finance (sin emps) quedan fuera

INNER JOIN (solo coincidencias):
  $Alice    | $Engineering  | $$75,000.00
  $Bob      | $Marketing    | $$55,000.00
  $Carol    | $Engineering  | $$82,000.00
  $David    | $HR           | $$48,000.00
  $Eva      | $Marketing    | $$61,000.00
